# Retrieval-Based Python Error Diagnosis System

This project builds a system that helps diagnose Python runtime errors by retrieving similar past errors.

Instead of training a classifier, the system converts error messages into vector representations and finds the most similar errors using cosine similarity.

### Pipeline

Python Error  
↓  
Text Cleaning  
↓  
TF-IDF Vectorization  
↓  
Cosine Similarity  
↓  
Top-3 Similar Errors  
↓  
Root Cause + Fix

In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
data = {
    "traceback": [
        "TypeError unsupported operand type for + int and str",
        "IndexError list index out of range",
        "KeyError name",
        "ValueError invalid literal for int",
        "NameError variable x is not defined",
        "AttributeError list object has no attribute split",
        "ZeroDivisionError division by zero",
        "ModuleNotFoundError no module named numpy",
        "FileNotFoundError no such file or directory",
        "IndentationError unexpected indent"
    ],

    "root_cause": [
        "Trying to add integer and string",
        "Accessing list index that does not exist",
        "Dictionary key missing",
        "Invalid conversion to integer",
        "Variable used before definition",
        "Using string method on list object",
        "Dividing a number by zero",
        "Library not installed",
        "File path incorrect",
        "Incorrect indentation in code"
    ],

    "fix": [
        "Convert string to integer before addition",
        "Check list length before accessing",
        "Check if key exists before accessing",
        "Ensure value contains numbers",
        "Define variable before using",
        "Convert list element to string",
        "Ensure denominator is not zero",
        "Install library using pip",
        "Check correct file path",
        "Fix indentation spacing"
    ]
}

df = pd.DataFrame(data)

df

,traceback,root_cause,fix
0,TypeError unsupported operand type for + int a...,Trying to add integer and string,Convert string to integer before addition
1,IndexError list index out of range,Accessing list index that does not exist,Check list length before accessing
2,KeyError name,Dictionary key missing,Check if key exists before accessing
3,ValueError invalid literal for int,Invalid conversion to integer,Ensure value contains numbers
4,NameError variable x is not defined,Variable used before definition,Define variable before using
5,AttributeError list object has no attribute split,Using string method on list object,Convert list element to string
6,ZeroDivisionError division by zero,Dividing a number by zero,Ensure denominator is not zero
7,ModuleNotFoundError no module named numpy,Library not installed,Install library using pip
8,FileNotFoundError no such file or directory,File path incorrect,Check correct file path
9,IndentationError unexpected indent,Incorrect indentation in code,Fix indentation spacing


## Text Preprocessing

In [3]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text


df["clean_traceback"] = df["traceback"].apply(clean_text)

df

,traceback,root_cause,fix,clean_traceback
0,TypeError unsupported operand type for + int a...,Trying to add integer and string,Convert string to integer before addition,typeerror unsupported operand type for int and...
1,IndexError list index out of range,Accessing list index that does not exist,Check list length before accessing,indexerror list index out of range
2,KeyError name,Dictionary key missing,Check if key exists before accessing,keyerror name
3,ValueError invalid literal for int,Invalid conversion to integer,Ensure value contains numbers,valueerror invalid literal for int
4,NameError variable x is not defined,Variable used before definition,Define variable before using,nameerror variable x is not defined
5,AttributeError list object has no attribute split,Using string method on list object,Convert list element to string,attributeerror list object has no attribute split
6,ZeroDivisionError division by zero,Dividing a number by zero,Ensure denominator is not zero,zerodivisionerror division by zero
7,ModuleNotFoundError no module named numpy,Library not installed,Install library using pip,modulenotfounderror no module named numpy
8,FileNotFoundError no such file or directory,File path incorrect,Check correct file path,filenotfounderror no such file or directory
9,IndentationError unexpected indent,Incorrect indentation in code,Fix indentation spacing,indentationerror unexpected indent


## TF-IDF Vectorization
Convert cleaned error messages into numerical vectors using TF-IDF.

In [4]:
vectorizer = TfidfVectorizer()

error_vectors = vectorizer.fit_transform(df["clean_traceback"])

print("Vector shape:", error_vectors.shape)

Vector shape: (10, 46)


## Error Retrieval System

Given a new Python error, the system retrieves the most similar past errors using cosine similarity.

In [5]:
def retrieve_similar_errors(query, top_k=3):

    # clean the input error
    query = clean_text(query)

    # convert to vector
    query_vector = vectorizer.transform([query])

    # compute similarity
    similarity_scores = cosine_similarity(query_vector, error_vectors)

    # get indices of top similar errors
    top_indices = similarity_scores[0].argsort()[-top_k:][::-1]

    # return matching rows
    results = df.iloc[top_indices]

    return results

## Test the Error Diagnosis System

In [6]:
query_error = "TypeError can only concatenate str to int"

results = retrieve_similar_errors(query_error)

results

,traceback,root_cause,fix,clean_traceback
0,TypeError unsupported operand type for + int a...,Trying to add integer and string,Convert string to integer before addition,typeerror unsupported operand type for int and...
3,ValueError invalid literal for int,Invalid conversion to integer,Ensure value contains numbers,valueerror invalid literal for int
8,FileNotFoundError no such file or directory,File path incorrect,Check correct file path,filenotfounderror no such file or directory


## Error Diagnosis Function
Display similar errors along with root causes and suggested fixes.

In [8]:
def diagnose_error(query):

    results = retrieve_similar_errors(query)

    print("Query Error:")
    print(query)
    print("\nTop Similar Errors:\n")

    for i, row in results.iterrows():

        print("Traceback:", row["traceback"])
        print("Root Cause:", row["root_cause"])
        print("Suggested Fix:", row["fix"])
        print("-" * 40)

## Run Error Diagnosis

In [9]:
diagnose_error("IndexError list index out of range")

Query Error:
IndexError list index out of range

Top Similar Errors:

Traceback: IndexError list index out of range
Root Cause: Accessing list index that does not exist
Suggested Fix: Check list length before accessing
----------------------------------------
Traceback: AttributeError list object has no attribute split
Root Cause: Using string method on list object
Suggested Fix: Convert list element to string
----------------------------------------
Traceback: FileNotFoundError no such file or directory
Root Cause: File path incorrect
Suggested Fix: Check correct file path
----------------------------------------


## Evaluation

To measure the performance of the retrieval system, we use **Top-3 Accuracy**.

For each error in the dataset:
1. Use the error as a query
2. Retrieve the Top-3 most similar errors
3. Check whether the correct error appears in the results

In [10]:
def evaluate_system():

    correct = 0

    for i, row in df.iterrows():

        query = row["traceback"]

        results = retrieve_similar_errors(query)

        retrieved_errors = results["traceback"].values

        if query in retrieved_errors:
            correct += 1

    accuracy = correct / len(df)

    print("Top-3 Accuracy:", accuracy)


evaluate_system()

Top-3 Accuracy: 1.0


## Expand Error Dataset
Add more Python runtime errors to improve the retrieval system.

In [12]:
more_data = {
    "traceback": [
        "TypeError unsupported operand type for minus int and str",
        "IndexError list assignment index out of range",
        "KeyError user",
        "ValueError could not convert string to float",
        "NameError variable y is not defined",
        "AttributeError dict object has no attribute append",
        "ZeroDivisionError float division by zero",
        "ModuleNotFoundError no module named pandas",
        "FileNotFoundError config file not found",
        "IndentationError expected an indented block",
        "TypeError object is not iterable",
        "ValueError math domain error",
        "KeyError id",
        "AttributeError tuple object has no attribute append",
        "IndexError tuple index out of range"
    ],

    "root_cause": [
        "Trying to subtract string from integer",
        "Assigning to an index that does not exist",
        "Dictionary key user missing",
        "Invalid conversion to float",
        "Variable y used before definition",
        "Using list method on dictionary",
        "Dividing floating number by zero",
        "Pandas library not installed",
        "Configuration file path incorrect",
        "Missing indentation after statement",
        "Looping over non iterable object",
        "Invalid mathematical input",
        "Dictionary key id missing",
        "Using list method on tuple",
        "Accessing tuple index that does not exist"
    ],

    "fix": [
        "Convert string to integer before subtraction",
        "Ensure index exists before assignment",
        "Check if key exists in dictionary",
        "Ensure value is numeric",
        "Define variable before using",
        "Use correct dictionary method",
        "Ensure denominator is not zero",
        "Install pandas using pip",
        "Check configuration file path",
        "Add indentation after block statements",
        "Ensure object is iterable",
        "Check mathematical input values",
        "Check if key id exists",
        "Use correct tuple operations",
        "Check tuple length before accessing"
    ]
}

df_more = pd.DataFrame(more_data)

df = pd.concat([df, df_more], ignore_index=True)

df.shape

(25, 4)

## Rebuild TF-IDF Vectors

Since we expanded the dataset, we regenerate TF-IDF vectors using the updated error dataset.

In [14]:
df["clean_traceback"] = df["traceback"].apply(clean_text)

df.head()

,traceback,root_cause,fix,clean_traceback
0,TypeError unsupported operand type for + int a...,Trying to add integer and string,Convert string to integer before addition,typeerror unsupported operand type for int and...
1,IndexError list index out of range,Accessing list index that does not exist,Check list length before accessing,indexerror list index out of range
2,KeyError name,Dictionary key missing,Check if key exists before accessing,keyerror name
3,ValueError invalid literal for int,Invalid conversion to integer,Ensure value contains numbers,valueerror invalid literal for int
4,NameError variable x is not defined,Variable used before definition,Define variable before using,nameerror variable x is not defined


In [15]:
vectorizer = TfidfVectorizer()

error_vectors = vectorizer.fit_transform(df["clean_traceback"])

print("Vector shape:", error_vectors.shape)

Vector shape: (25, 69)


## Test Retrieval System on Expanded Dataset

In [16]:
diagnose_error("TypeError cannot add string and integer")

Query Error:
TypeError cannot add string and integer

Top Similar Errors:

Traceback: TypeError unsupported operand type for + int and str
Root Cause: Trying to add integer and string
Suggested Fix: Convert string to integer before addition
----------------------------------------
Traceback: TypeError unsupported operand type for minus int and str
Root Cause: Trying to subtract string from integer
Suggested Fix: Convert string to integer before subtraction
----------------------------------------
Traceback: ValueError could not convert string to float
Root Cause: Invalid conversion to float
Suggested Fix: Ensure value is numeric
----------------------------------------


## Save Dataset
Save the error dataset so it can be reused without recreating it.

In [17]:
df.to_csv("python_errors_dataset.csv", index=False)

print("Dataset saved successfully.")

Dataset saved successfully.


## Load Dataset

In [18]:
df = pd.read_csv("python_errors_dataset.csv")

df.head()

,traceback,root_cause,fix,clean_traceback
0,TypeError unsupported operand type for + int a...,Trying to add integer and string,Convert string to integer before addition,typeerror unsupported operand type for int and...
1,IndexError list index out of range,Accessing list index that does not exist,Check list length before accessing,indexerror list index out of range
2,KeyError name,Dictionary key missing,Check if key exists before accessing,keyerror name
3,ValueError invalid literal for int,Invalid conversion to integer,Ensure value contains numbers,valueerror invalid literal for int
4,NameError variable x is not defined,Variable used before definition,Define variable before using,nameerror variable x is not defined


## Conclusion

In this project, we built a retrieval-based system to diagnose Python runtime errors.

Key steps in the pipeline:

1. Created a dataset of Python runtime errors with root causes and fixes
2. Cleaned error messages using basic text preprocessing
3. Converted error messages into numerical vectors using TF-IDF
4. Computed similarity between errors using cosine similarity
5. Retrieved the Top-3 most similar errors for diagnosis
6. Evaluated the system using Top-3 Accuracy

This approach demonstrates how information retrieval techniques can assist developers in debugging Python errors efficiently.